# Imports

In [1]:
import os
import spacy
import gensim
import pickle
import numpy as np
from tqdm import notebook
from pymystem3 import Mystem
from nltk import sent_tokenize
from ruwordnet import RuWordNet
from collections import Counter
from collections import namedtuple
from sklearn.neighbors import KDTree
from nltk.tokenize import word_tokenize
from gensim.models.phrases import Phrases

In [5]:
spacy_nlp = spacy.load('ru_core_news_sm')

In [6]:
RU_STOP_WORDS = spacy_nlp.Defaults.stop_words
print(RU_STOP_WORDS)
del spacy_nlp

{'самим', 'наиболее', 'если', 'именно', 'многом', 'ваши', 'отовсюду', 'могло', 'самое', 'гав', 'оп', 'насчет', 'будет', 'от', 'прочего', 'самому', 'оттого', 'поскольку', 'все', 'почему', 'весь', 'гораздо', 'те', 'этой', 'просто', 'ух', 'будешь', 'д', 'слишком', 'ф', 'р', 'та', 'этому', 'твоя', 'ныне', 'здесь', 'того', 'касательно', 'едва', 'наипаче', 'вместо', 'напрямую', 'очевидно', 'были', 'под', 'эдак', 'дальше', 'всему', 'потомушта', 'ша', 'той', 'пусть', 'чьё', 'коль', 'о', 'бываю', 'скоро', 'покамест', 'отнелижа', 'мной', 'многое', 'некоторых', 'вами', 'неё', 'ау', 'эту', 'неоткуда', 'быть', 'да', 'наконец', 'её', 'ближайшие', 'есть', 'каждый', 'нашею', 'якоже', 'нашем', 'моим', 'ему', 'х', 'отчего', 'их', 'фу', 'каждые', 'бывали', 'которой', 'имело', 'всею', 'итак', 'имъ', 'ой', 'оба', 'меж', 'ибо', 'между', 'спокону', 'сызнова', 'безусловно', 'своём', 'ли', 'томах', 'какого', 'можем', 'необходимым', 'такого', 'ну', 'перед', 'лишь', 'затем', 'паче', 'сие', 'многому', 'нечем', 'п

In [7]:
nlp = Mystem(disambiguation=True)

In [9]:
wn = RuWordNet()

In [10]:
model_path = '214/model.model'
fasttext_model = gensim.models.fasttext.FastTextKeyedVectors.load(model_path)

In [11]:
with open('GeoWac_tokens_words_tree_synset_ids.pickle', 'rb') as handle:
    words, tree, synset_ids = pickle.load(handle)

# Define annotating functions

In [12]:
def vectorize(text):
    text = str(text)
    vec = np.sum([fasttext_model[word] for word in ' '.join(word_tokenize(text, language='russian')).lower().split()], axis=0)
    vec /= sum(vec**2) ** 0.5 
    return vec

In [13]:
def distance2vote(d, a=3, b=5):
    sim = np.maximum(0, 1 - d**2/2)
    return np.exp(-d**a) * sim **b

In [14]:
UNK_WORD_TO_SYNSETS = {}
def get_WordNet_synsets(token, unknown_words=UNK_WORD_TO_SYNSETS, top_n_senses=10):
    # Если токен, которого нет в WN, уже встречался, достаем его из "кэша"
    if token in unknown_words:
        return unknown_words[token]
        
    votes = Counter()
    dists, ids = tree.query(vectorize(token).reshape(1, -1), k=100)
    for idx, distance in zip(ids[0], dists[0]):
        for hyper in wn[synset_ids[idx]].hypernyms:
            votes[hyper.id] += distance2vote(distance)
    out = []
    for sid, score in votes.most_common(top_n_senses):
        out.append(wn[sid])
    unknown_words[token] = out # сохраняем токен в "кэш", чтобы потом не вычислять его снова и ускорить работу
    return out

## ВАЖНО
При подъеме по иерархии можно уткнуться в потолок гиперонимов, когда их просто нет. 
Синсет есть, первый гипероним есть, а второго уже нет.

Поэтому предлагается следующая реализация:
- Если 2й гипероним не найден, возвращается функция дизамбигуации по 1ому гиперониму
- Если 1й гипероним не найден, возвращается функция дизамбигуации по исходному синсету
- Если дизамбигуация по синсету не дает результатов, возвращается namedtuple(object=None, title='ERROR'), чтобы избежать конфликтов

In [15]:
def get_candidate_scores(text_vector:np.ndarray, candidates:list):
    candidate_vectors = [
        vectorize(' '.join(w for s in c.senses for w in s.name.lower().split()))
        for c in candidates]
    return [np.dot(text_vector, v) for v in candidate_vectors]

In [16]:
def disambiguate_word_fasttext(word:str, context:list):
    # Непосредственные синсеты слова
    try:
        candidates = [s.synset for s in wn[word]]
    except KeyError:
        candidates = get_WordNet_synsets(word)
        
    # Вычисляем метрики для синсетов, ищем лучший
    text_vector = vectorize(' '.join([tk for tk in context if tk != word]))
    scores = get_candidate_scores(text_vector, candidates)
    
    if not scores:
        out = namedtuple('OUT', ['title', ])
        return out(title='ERROR')
    max_score_index = scores.index(max(scores))
    return candidates[max_score_index]

In [17]:
def disambiguate_word_fasttext_hyper1(word:str, context:list):
    # Берем синстет слова, потом от него поднимаемся на 1 шаг наверх (первый гипероним)
    try:
        candidates = [first_hypernym for sense in wn[word] for first_hypernym in sense.synset.hypernyms]
    except KeyError:
        candidates =  [first_hypernym for synset in get_WordNet_synsets(word) for first_hypernym in synset.hypernyms]
        
    text_vector = vectorize(' '.join([tk for tk in context if tk != word]))
    scores = get_candidate_scores(text_vector, candidates)

    if not scores:
        return disambiguate_word_fasttext(word, context)
    max_score_index = scores.index(max(scores))
    return candidates[max_score_index]

In [18]:
def disambiguate_word_fasttext_post_hyper1(word:str, context:list):
    # Непосредственные синсеты слова
    try:
        candidates = [s.synset for s in wn[word]]
    except KeyError:
        candidates = get_WordNet_synsets(word)
    
    # Вычисляем метрики для синсетов, ищем лучший  
    text_vector = vectorize(' '.join([tk for tk in context if tk != word]))
    scores = get_candidate_scores(text_vector, candidates)
    
    if not scores:
        out = namedtuple('OUT', ['title', ])
        return out(title='ERROR')

    # У лучшего синсета берем гиперонимы, выбираем лучший из них
    max_score_index = scores.index(max(scores))
    hyper1_candidates = candidates[max_score_index].hypernyms
    hyper1_scores = get_candidate_scores(text_vector, hyper1_candidates)
    # Если с гиперонимами не получилось, возвращаем просто лучший синсет
    if not hyper1_scores:
        return candidates[max_score_index]
        
    # Иначе возвращаем лучший гипероним лучшего синсета
    max_score_index = hyper1_scores.index(max(hyper1_scores))
    return hyper1_candidates[max_score_index]

In [19]:
def disambiguate_word_fasttext_hyper2(word:str, context:list):
    # Берем синстет слова, потом от него поднимаемся на 2 шага наверх (второй гипероним)
    try:
        synsets = [s.synset for s in wn[word]]
    except KeyError:
        synsets = get_WordNet_synsets(word)
    
    candidates = []
    for synset in synsets:
        for first_hypernym in synset.hypernyms:
            for second_hypernym in first_hypernym.hypernyms:
                candidates.append(second_hypernym)

    text_vector = vectorize(' '.join([tk for tk in context if tk != word]))
    scores = get_candidate_scores(text_vector, candidates)
    
    if not scores:
        return disambiguate_word_fasttext_hyper1(word, context)
    max_score_index = scores.index(max(scores))
    return candidates[max_score_index]

In [20]:
def disambiguate_word_fasttext_post_hyper2(word:str, context:list):
    # Непосредственные синсеты слова
    try:
        candidates = [s.synset for s in wn[word]]
    except KeyError:
        candidates = get_WordNet_synsets(word)
    
    # Вычисляем метрики для синсетов, ищем лучший  
    text_vector = vectorize(' '.join([tk for tk in context if tk != word]))
    scores = get_candidate_scores(text_vector, candidates)
    
    if not scores:
        out = namedtuple('OUT', ['title', ])
        return out(title='ERROR')
    max_score_index = scores.index(max(scores))

    # У лучшего синсета берем гиперонимы, выбираем лучший из них
    hyper1_candidates = candidates[max_score_index].hypernyms
    hyper1_scores = get_candidate_scores(text_vector, hyper1_candidates)
    # Если с гиперонимами не получилось, возвращаем просто лучший синсет
    if not hyper1_scores:
        return candidates[max_score_index]
        
    # Иначе обращаемся к поиску лучшего второго гиперонима
    max_score_index_h1 = hyper1_scores.index(max(hyper1_scores))
    hyper2_candidates = hyper1_candidates[max_score_index_h1].hypernyms
    hyper2_scores = get_candidate_scores(text_vector, hyper2_candidates)
    # Если не получилось со вторыми гиперонимами, возвращаем лучший первый гипероним
    if not hyper2_scores:
        return hyper1_candidates[max_score_index_h1]
    max_score_index_h2 = hyper2_scores.index(max(hyper2_scores))
    return hyper2_candidates[max_score_index_h2]

In [21]:
def disambiguate_word_fasttext_many(word:str, context:list):
    # Берем синстет слова, потом от него поднимаемся на 1-2-3 шага наверх, все сохраняем.
    # Как видно ниже, в выдачу попадают непосредственные синсеты, а также 1 и 2 гиперонимы.
    # Но это только два примера. Можно изучать подробнее
    try:
        synsets = [s.synset for s in wn[word]]
    except KeyError:
        synsets = get_WordNet_synsets(word)
    
    candidates = []
    for synset in synsets:
        candidates.append(synset)
        for first_hypernym in synset.hypernyms:
            candidates.append(first_hypernym)
            for second_hypernym in first_hypernym.hypernyms:
                candidates.append(second_hypernym)
                for third_hypernym in second_hypernym.hypernyms:
                    candidates.append(third_hypernym) 
    
    text_vector = vectorize(' '.join([tk for tk in context if tk != word]))
    scores = get_candidate_scores(text_vector, candidates)
    
    if not scores:
        out = namedtuple('OUT', ['title', ])
        return out(title='ERROR')
    max_score_index = scores.index(max(scores))
    return candidates[max_score_index]

# Read text to analyze

In [22]:
# Читаем сюда текст, можно хоть весь учебник. 
# Анализ по предложениям, поэтому работать будет в любом случае о-о-о-очень долго (1 параграф// 186 предложений // 2237 слов // 12,5 минут)
text = """
Становление и развитие русской культуры - это длительный процесс. Государство создало благоприятные условия для развития культуры. Доказательством этого являлся разительный подъем культуры Киевской Руси. На культуру народа воздействие оказывает географическая среда, нравы, традиции, все культурное наследие, доставшееся от предыдущих поколений. Политическое объединение восточнославянских племен способствовало их этнической консолидации и духовному единению.
"""

# Split text into sentences

In [23]:
raw_sentences = sent_tokenize(text, language='russian')

# Lemmatize words inside sentences and keep only isalpha tokens

In [24]:
lemmatized_sentences = []
for sentence in notebook.tqdm(raw_sentences):
    lemmatized_sentences.append([lemma for lemma in nlp.lemmatize(sentence) if lemma.isalpha()])
# Может работать не оптимально быстро, потому что фильтрация if lemma.isalpha() и потому что анализ по предложениям

  0%|          | 0/5 [00:00<?, ?it/s]

# Define a model for collocation exctracton. Perform collocation extraction

In [25]:
# Модель для получения коллокаций; 
# можно итеративно загружать данные и получать коллокации с большим количесвом слов 
# (надо изучить на предмет выделения многословных терминов, в дополнение к словарному методу)
phrases = Phrases(
    lemmatized_sentences, 
    min_count=1, # Ignore all words and bigrams with total collected count lower than this value.
    threshold=1, # Represent a score threshold for forming the phrases (higher means fewer phrases). 
    # A phrase of words a followed by b is accepted if the score of the phrase is greater than threshold. 
    # Heavily depends on concrete scoring-function, see the scoring parameter.
    
    connector_words=RU_STOP_WORDS # Надо бы указать список союзов, предлогов и тд: для английского это gensim.models.phrases.ENGLISH_CONNECTOR_WORDS
    # 'the', 'without', 'with', 'from', 'a', 'of', 'at', 'for', 'to', 'by', 'and', 'on', 'an', 'in', 'or'
)
print(*sorted(phrases.export_phrases().items(), key=lambda x: x[1]), sep='\n')

In [26]:
lemmatized_sentences_with_collocations = [phrases[sentence] for sentence in lemmatized_sentences]

# Perform WSD

In [27]:
annoated_simple = []
for sentence in notebook.tqdm(lemmatized_sentences_with_collocations):
    annotated_sent_simple = []
    for token in sentence:
        if token in RU_STOP_WORDS:
            annotated_sent_simple.append(token)
        else:
            annotated_sent_simple.append(f'{token}%{disambiguate_word_fasttext(token, sentence).title}')
    annoated_simple.append(annotated_sent_simple)

  0%|          | 0/5 [00:00<?, ?it/s]

In [28]:
annotated_hyper1 = []
for sentence in notebook.tqdm(lemmatized_sentences_with_collocations):
    annotated_sent_h1 = []
    for token in sentence:
        if token in RU_STOP_WORDS:
            annotated_sent_h1.append(token)
        else:
            annotated_sent_h1.append(f'{token}%{disambiguate_word_fasttext_hyper1(token, sentence).title}')
    annotated_hyper1.append(annotated_sent_h1)

  0%|          | 0/5 [00:00<?, ?it/s]

In [29]:
annotated_post_hyper1 = []
for sentence in notebook.tqdm(lemmatized_sentences_with_collocations):
    annotated_sent_post_h1 = []
    for token in sentence:
        if token in RU_STOP_WORDS:
            annotated_sent_post_h1.append(token)
        else:
            annotated_sent_post_h1.append(f'{token}%{disambiguate_word_fasttext_post_hyper1(token, sentence).title}')
    annotated_post_hyper1.append(annotated_sent_post_h1)

  0%|          | 0/5 [00:00<?, ?it/s]

In [30]:
annotated_hyper2 = []
for sentence in notebook.tqdm(lemmatized_sentences_with_collocations):
    annotated_sent_h2 = []
    for token in sentence:
        if token in RU_STOP_WORDS:
            annotated_sent_h2.append(token)
        else:
            annotated_sent_h2.append(f'{token}%{disambiguate_word_fasttext_hyper2(token, sentence).title}')
    annotated_hyper2.append(annotated_sent_h2)

  0%|          | 0/5 [00:00<?, ?it/s]

In [31]:
annotated_post_hyper2 = []
for sentence in notebook.tqdm(lemmatized_sentences_with_collocations):
    annotated_sent_post_h2 = []
    for token in sentence:
        if token in RU_STOP_WORDS:
            annotated_sent_post_h2.append(token)
        else:
            annotated_sent_post_h2.append(f'{token}%{disambiguate_word_fasttext_post_hyper2(token, sentence).title}')
    annotated_post_hyper2.append(annotated_sent_post_h2)

  0%|          | 0/5 [00:00<?, ?it/s]

In [32]:
annotated_many = []
for sentence in notebook.tqdm(lemmatized_sentences_with_collocations):
    annotated_sent_many = []
    for token in sentence:
        if token in RU_STOP_WORDS:
            annotated_sent_many.append(token)
        else:
            annotated_sent_many.append(f'{token}%{disambiguate_word_fasttext_many(token, sentence).title}')  
    annotated_many.append(annotated_sent_many)

  0%|          | 0/5 [00:00<?, ?it/s]

# Compare results

In [33]:
import pandas as pd

In [34]:
df_lemmas = pd.DataFrame([i for s in lemmatized_sentences_with_collocations for i in s], columns=['lemma'])
df_s = pd.DataFrame([token for sentence in annoated_simple for token in sentence], columns=['simple'])
df_h1 = pd.DataFrame([token for sentence in annotated_hyper1 for token in sentence], columns=['hyper1'])
df_post_h1 = pd.DataFrame([token for sentence in annotated_post_hyper1 for token in sentence], columns=['post_hyper1'])
df_h2 = pd.DataFrame([token for sentence in annotated_hyper2 for token in sentence], columns=['hyper2'])
df_post_h2 = pd.DataFrame([token for sentence in annotated_post_hyper2 for token in sentence], columns=['post_hyper2'])
df_many = pd.DataFrame([token for sentence in annotated_many for token in sentence], columns=['many'])

In [39]:
DF = pd.concat([df_lemmas, df_s, df_h1, df_post_h1, df_h2, df_post_h2, df_many], axis=1)
DF['in_WN'] = ~DF['lemma'].isin(set(UNK_WORD_TO_SYNSETS))

In [42]:
DF.head(60)

,lemma,simple,hyper1,post_hyper1,hyper2,post_hyper2,many,in_WN
0,становление,"становление%ФОРМИРОВАТЬСЯ, СЛАГАТЬСЯ (ПРИОБРЕТ...","становление%ИЗМЕНИТЬСЯ, ИЗМЕНЕНИЕ","становление%ИЗМЕНИТЬСЯ, ИЗМЕНЕНИЕ","становление%ПРОЦЕСС, СМЕНА СОСТОЯНИЙ","становление%ПРОЦЕСС, СМЕНА СОСТОЯНИЙ","становление%ФОРМИРОВАТЬСЯ, СЛАГАТЬСЯ (ПРИОБРЕТ...",True
1,и,и,и,и,и,и,и,True
2,развитие,развитие%РАЗВИТИЕ,"развитие%ИЗМЕНИТЬСЯ, ИЗМЕНЕНИЕ","развитие%ИЗМЕНИТЬСЯ, ИЗМЕНЕНИЕ","развитие%ПРОЦЕСС, СМЕНА СОСТОЯНИЙ","развитие%ПРОЦЕСС, СМЕНА СОСТОЯНИЙ",развитие%РАЗВИТИЕ,True
3,русский,русский%РУССКИЕ,русский%ТИТУЛЬНЫЙ НАРОД,русский%ТИТУЛЬНЫЙ НАРОД,русский%ЧЕЛОВЕК,русский%ЭТНИЧЕСКИЕ НАРОДЫ,русский%ЧЕЛОВЕК,True
4,культура,культура%КУЛЬТУРА (СФЕРА ДЕЯТЕЛЬНОСТИ),культура%СФЕРА ДЕЯТЕЛЬНОСТИ,культура%СФЕРА ДЕЯТЕЛЬНОСТИ,"культура%ЗАНЯТИЕ, ДЕЯТЕЛЬНОСТЬ","культура%ЗАНЯТИЕ, ДЕЯТЕЛЬНОСТЬ",культура%КУЛЬТУРА (СФЕРА ДЕЯТЕЛЬНОСТИ),True
5,это,это,это,это,это,это,это,True
6,длительный,"длительный%ДЛИТЕЛЬНЫЙ, ДОЛГИЙ","длительный%БОЛЬШОЙ, ЗНАЧИТЕЛЬНЫЙ","длительный%БОЛЬШОЙ, ЗНАЧИТЕЛЬНЫЙ","длительный%СВОЙСТВО, ХАРАКТЕРИСТИКА","длительный%СВОЙСТВО, ХАРАКТЕРИСТИКА","длительный%ДЛИТЕЛЬНЫЙ, ДОЛГИЙ",True
7,процесс,"процесс%ПРОЦЕСС, СМЕНА СОСТОЯНИЙ",процесс%ПРОИСХОДЯЩАЯ СУЩНОСТЬ,процесс%ПРОИСХОДЯЩАЯ СУЩНОСТЬ,"процесс%ОПРЕДЕЛИТЬ, ВЫЯСНИТЬ",процесс%ПРОИСХОДЯЩАЯ СУЩНОСТЬ,"процесс%УЗНАТЬ, ПОЛУЧИТЬ СВЕДЕНИЯ",True
8,государство,государство%ГОСУДАРСТВО,государство%ПУБЛИЧНО-ПРАВОВОЕ ОБРАЗОВАНИЕ,государство%ПУБЛИЧНО-ПРАВОВОЕ ОБРАЗОВАНИЕ,государство%СУБЪЕКТ ПРАВА,государство%СУБЪЕКТ ПРАВА,государство%ПУБЛИЧНО-ПРАВОВОЕ ОБРАЗОВАНИЕ,True
9,создавать,"создавать%ОБЕСПЕЧИТЬ, СОЗДАТЬ УСЛОВИЯ","создавать%ОБУСЛАВЛИВАТЬ, СПОСОБСТВОВАТЬ",создавать%СОДЕЙСТВИЕ,"создавать%УЧАСТИЕ, УЧАСТВОВАТЬ","создавать%УЧАСТИЕ, УЧАСТВОВАТЬ","создавать%ОБЕСПЕЧИТЬ, СОЗДАТЬ УСЛОВИЯ",True


In [ ]:
DF.to_excel('WSD_RuCultHist_geowac_tokens.xlsx')